# Export saved seed results — ไม่รวม dataset และ input cache

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/nih -xray/Notebook_3_release/runs/release_v4/seed_42")
except ImportError:
    RUN_ROOT = Path.cwd() / "runs/release_v4/seed_42"

INCLUDE_CHECKPOINTS = False  # เก็บ weights บน Drive แล้ว; เปิดได้หากต้องการ export ด้วย


In [ ]:
"""Export one measured seed; original dataset and runtime cache are excluded."""
from pathlib import Path
import argparse
import zipfile


def export_results(run_root, destination=None, include_checkpoints=False):
    run_root = Path(run_root).resolve()
    if not (run_root / "shared/artifacts/experiment.json").is_file():
        raise FileNotFoundError("Select an existing measured seed directory")
    target = Path(destination or run_root.parent / f"{run_root.name}_results.zip").resolve()
    if target.is_relative_to(run_root):
        raise ValueError("Export destination must be outside the run directory")
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(".tmp.zip")
    with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED) as archive:
        for p in sorted(run_root.rglob("*")):
            if not p.is_file() or p.suffix in (".tmp", ".zip"):
                continue
            if not include_checkpoints and p.suffix in (".pt", ".pth", ".ckpt"):
                continue
            archive.write(p, run_root.name + "/" + p.relative_to(run_root).as_posix())
    temp.replace(target)
    print("Exported", target, "| checkpoints:", include_checkpoints)
    return target



export_results(RUN_ROOT, include_checkpoints=INCLUDE_CHECKPOINTS)
